# NeerNiti — Site Recommendation Model (v23.0)
### High-Volume Extraction, Checkpointing, and Robust Retries

**Key Additions:**
1. **High-Volume Endpoint:** Earth Engine now initializes using the `earthengine-highvolume.googleapis.com` endpoint meant for bulk parallel extraction.
2. **Robust Checkpointing:** GEE extractions are appended to a `.csv` immediately after each chunk finishes. If a crash or timeout occurs, restarting the notebook will instantly resume from the exact well it left off at.
3. **Adaptive Splitting & ThreadPool:** Added multi-threading with adaptive fallback. If GEE throws a timeout on a chunk, it recursively splits the chunk in half and tries again instead of failing.
4. **Optimized Distance Transform:** Capped the `fastDistanceTransform` to `128` pixels (instead of 512) to massively speed up the GEE backend computation without impacting the tree model's ability to threshold distance.

In [ ]:
!pip install -q earthengine-api catboost scikit-learn pandas numpy matplotlib tqdm google-api-python-client seaborn optuna statsmodels shap

import os, json, time, warnings
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.calibration import CalibratedClassifierCV
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, brier_score_loss, classification_report, confusion_matrix, roc_curve
from datetime import datetime, timezone
import ee
from google.oauth2 import service_account
from concurrent.futures import ThreadPoolExecutor, as_completed
import joblib

warnings.filterwarnings('ignore')
RUNTIME = 'kaggle' if os.path.exists('/kaggle') else 'colab'
print(f'Runtime: {RUNTIME.upper()}')

In [ ]:
if RUNTIME == 'kaggle':
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    creds = service_account.Credentials.from_service_account_info(
        json.loads(_secrets.get_secret('GEE_SERVICE_ACCOUNT_KEY')),
        scopes=['https://www.googleapis.com/auth/earthengine', 'https://www.googleapis.com/auth/drive']
    )
    ee.Initialize(creds, project=_secrets.get_secret('GEE_PROJECT_ID'), opt_url='https://earthengine-highvolume.googleapis.com')
else:
    ee.Authenticate()
    ee.Initialize(project=input('Enter GEE Project ID: ').strip(), opt_url='https://earthengine-highvolume.googleapis.com')
print('GEE Authenticated with High-Volume Endpoint.')

In [ ]:
TEL_2021_CSV_PATH = '/kaggle/input/datasets/chinmay302/neerniti/gwl_tel_6_hourly_upgw_up_2021_2025.csv'
CGWB_2021_CSV_PATH = '/kaggle/input/datasets/chinmay302/neerniti/gwl_tel_6_hourly_cgwb_up_2021_2025.csv'
GEE_CACHE_FILE = '/kaggle/working/up_gee_cache.csv'
GEE_BUFFER_M = 500
VERSION_ID = 'neerniti_up_model'
RANDOM_SEEDS = [42, 100, 2024]

STAGE_OF_EXTRACTION = {
    'agra': 116.8, 'aligarh': 72.51, 'ambedkar nagar': 62.02, 'amethi': 61.06, 'amroha': 96.4, 
    'auraiya': 49.08, 'aurraiya': 49.08, 'ayodhya': 63.13, 'azamgarh': 68.58, 'baghpat': 97.9, 
    'bahraich': 54.09, 'ballia': 62.73, 'balrampur': 56.46, 'banda': 65.03, 'barabanki': 61.16, 
    'bareilly': 67.94, 'bareili': 67.94, 'basti': 62.83, 'bijnor': 69.37, 'bijnaur': 69.37, 
    'budaun': 77.13, 'bulandshahr': 91.38, 'bulandshahar': 91.38, 'chandauli': 52.61, 
    'chitrakoot': 82.66, 'deoria': 59.15, 'etah': 79.73, 'etawah': 42.71, 'farrukhabad': 73.27, 
    'fatehpur': 70.27, 'firozabad': 112.0, 'gautam buddha nagar': 101.52, 'g.b. nagar': 101.52, 
    'ghaziabad': 113.97, 'ghazipur': 60.83, 'gonda': 61.17, 'gorakhpur': 59.76, 'hamirpur': 68.1, 
    'hapur': 100.18, 'hardoi': 58.32, 'hathras': 95.57, 'jalaun': 50.47, 'jaunpur': 71.02, 
    'jhansi': 57.79, 'kannauj': 64.74, 'kanpur dehat': 72.96, 'kanpur nagar': 76.06, 'kasganj': 73.51, 
    'kaushambi': 76.52, 'kushinagar': 42.37, 'kushi nagar': 42.37, 'lakhimpur kheri': 72.1, 
    'kheri': 72.1, 'lakhimpur': 72.1, 'lalitpur': 79.58, 'lucknow': 66.49, 'mahrajganj': 60.27, 
    'maharajganj': 60.27, 'mahoba': 93.57, 'mainpuri': 69.24, 'mathura': 73.1, 'mau': 65.68, 
    'meerut': 77.69, 'mirzapur': 60.31, 'moradabad': 85.03, 'muzaffarnagar': 72.57, 'pilibhit': 57.2, 
    'pratapgarh': 74.75, 'prayagraj': 74.19, 'rae bareli': 55.68, 'rampur': 72.11, 
    'sant kabir nagar': 66.18, 's.kabir nagar': 66.18, 'sant ravidas nagar': 73.1, 's.ravidas': 73.1, 
    'saharanpur': 106.45, 'shahjahanpur': 60.84, 'sambhal': 85.93, 'shambhal': 85.93, 'shamli': 104.18, 
    'shravasti': 58.93, 'shrawasti': 58.93, 'siddharthnagar': 68.2, 'siddharth': 68.2, 
    'sitapur': 57.34, 'sonbhadra': 64.67, 'sultanpur': 60.28, 'unnao': 60.67, 'varanasi': 91.21,
    'praygraj': 74.19, 'allahabad': 74.19, 'bhadohi': 73.1, 'raibareli': 55.68, 
    'amethi (c.s.m. nagar)': 61.06, 'kansiram nagar': 73.51, 'shahjhanpur': 60.84, 
    'buland shahar': 91.38, 'amroha (j.p.nagar)': 96.4, 'jyotiba phule nagar': 96.4
}

def normalize_district(d):
    d = str(d).strip().lower()
    aliases = {'bareili':'bareilly', 'bijnaur':'bijnor', 'praygraj':'prayagraj', 'allahabad':'prayagraj',
               'buland shahar':'bulandshahr', 'bulandshahar':'bulandshahr', 'shrawasti':'shravasti',
               'shambhal':'sambhal', 'siddharth':'siddharthnagar', 'maharajganj':'mahrajganj',
               'kushi nagar':'kushinagar', 'raibareli':'rae bareli', 'kheri':'lakhimpur kheri',
               'lakhimpur':'lakhimpur kheri', 'aurraiya':'auraiya', 's.kabir nagar':'sant kabir nagar',
               's.ravidas':'bhadohi', 'sant ravidas nagar':'bhadohi', 'g.b. nagar':'gautam buddha nagar',
               'amethi (c.s.m. nagar)':'amethi', 'amroha (j.p.nagar)':'amroha', 'jyotiba phule nagar':'amroha',
               'kansiram nagar':'kasganj', 'shahjhanpur':'shahjahanpur'}
    return aliases.get(d, d)

In [ ]:
HYDRO_ZONES = {
    # Western UP (Over-Exploited / Sugarcane / Heavy Tubewell)
    'agra': 'Western', 'firozabad': 'Western', 'hathras': 'Western', 'aligarh': 'Western',
    'baghpat': 'Western', 'meerut': 'Western', 'ghaziabad': 'Western', 'gautam buddha nagar': 'Western',
    'bulandshahr': 'Western', 'saharanpur': 'Western', 'shamli': 'Western', 'muzaffarnagar': 'Western',
    'mathura': 'Western', 'moradabad': 'Western',
    
    # Central/Eastern UP (Critical / Urban / Domestic Pumping)
    'kanpur nagar': 'Central', 'lucknow': 'Central', 'bareilly': 'Central', 'varanasi': 'Central',
    'jaunpur': 'Central', 'prayagraj': 'Central', 'fatehpur': 'Central', 'kannauj': 'Central',
    'farrukhabad': 'Central', 'sambhal': 'Central', 'budaun': 'Central',
    
    # Bundelkhand (Hard-Rock Granite / Geographic Distress)
    'banda': 'Bundelkhand', 'chitrakoot': 'Bundelkhand', 'mahoba': 'Bundelkhand',
    'lalitpur': 'Bundelkhand', 'hamirpur': 'Bundelkhand', 'jalaun': 'Bundelkhand', 'sonbhadra': 'Bundelkhand',
    
    # Terai & Eastern Belt (Safe / High Absorption / Himalayan Strip)
    'siddharthnagar': 'Terai', 'kushinagar': 'Terai', 'deoria': 'Terai', 'shravasti': 'Terai',
    'balrampur': 'Terai', 'bahraich': 'Terai'
}


In [ ]:
def process_telemetry():
    print("Loading UP Telemetry datasets (Executing Strict QC)...")
    if not os.path.exists(TEL_2021_CSV_PATH): raise FileNotFoundError("Missing dataset")
    df_upgw = pd.read_csv(TEL_2021_CSV_PATH, low_memory=False)
    if os.path.exists(CGWB_2021_CSV_PATH):
        print("Merging CGWB Telemetry...")
        df_cgwb = pd.read_csv(CGWB_2021_CSV_PATH, low_memory=False)
        df = pd.concat([df_upgw, df_cgwb], ignore_index=True)
    else:
        df = df_upgw
    df['Date'] = pd.to_datetime(df['Data Acquisition Time'], errors='coerce', format='mixed', dayfirst=True)
    df = df.dropna(subset=['Date'])
    df = df[df['Date'].dt.year <= 2025]
    print(f"Wells initially: {df[['Latitude', 'Longitude']].drop_duplicates().shape[0]}")
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    lvl_col = [c for c in df.columns if 'meter' in c.lower()][0]
    df[lvl_col] = pd.to_numeric(df[lvl_col], errors='coerce').abs()
    
    df = df[df[lvl_col] > 0]
    df['Season'] = np.where(df['Month'].isin([4, 5]), 'Pre', np.where(df['Month'].isin([10, 11]), 'Post', 'Other'))
    df = df[df['Season'] != 'Other']
    
    agg_df = df.groupby(['Latitude', 'Longitude', 'Year', 'District', 'Season'])[lvl_col].agg(['median', 'std', 'count']).reset_index()
    print(f"Well-years before QC: {len(agg_df)}")
    print(f"Wells before QC: {agg_df[['Latitude', 'Longitude']].drop_duplicates().shape[0]}")
    agg_df = agg_df[(agg_df['count'] >= 20) & (agg_df['std'] > 0)]
    print(f"Well-years after count>=20 & std>0: {len(agg_df)}")
    print(f"Wells after count/std QC: {agg_df[['Latitude', 'Longitude']].drop_duplicates().shape[0]}")
    
    pre = agg_df[agg_df['Season'] == 'Pre'].rename(columns={'median': 'pre_monsoon_m'})
    post = agg_df[agg_df['Season'] == 'Post'].rename(columns={'median': 'post_monsoon_m'})
    
    panel_df = pd.merge(pre[['Latitude', 'Longitude', 'Year', 'District', 'pre_monsoon_m']], 
                        post[['Latitude', 'Longitude', 'Year', 'District', 'post_monsoon_m']], 
                        on=['Latitude', 'Longitude', 'Year', 'District'], how='inner')
    
    panel_df['District'] = panel_df['District'].apply(normalize_district)
    locations = panel_df[['Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)
    locations['well_id'] = locations.apply(lambda r: f"{r.Latitude:.5f}_{r.Longitude:.5f}", axis=1)
    panel_df = panel_df.merge(locations, on=['Latitude', 'Longitude'], how='left')
    
    year_counts = panel_df.groupby('well_id').size()
    valid_wells = year_counts[year_counts >= 2].index
    print(f"Wells surviving >=2 years rule: {len(valid_wells)}")
    panel_df = panel_df[panel_df['well_id'].isin(valid_wells)].copy()
    panel_df['recharge_potential'] = panel_df['pre_monsoon_m'] - panel_df['post_monsoon_m']
    
    bhabar_terai = ['saharanpur', 'bijnor', 'moradabad', 'rampur', 'bareilly', 'pilibhit', 'lakhimpur kheri', 'bahraich', 'shravasti', 'balrampur', 'siddharthnagar', 'mahrajganj']
    bundelkhand = ['jhansi', 'lalitpur', 'jalaun', 'hamirpur', 'mahoba', 'banda', 'chitrakoot', 'mirzapur', 'sonbhadra']
    def get_hydro_zone(d): return 0 if d in bhabar_terai else 2 if d in bundelkhand else 1
    panel_df['pdf_hydro_zone'] = panel_df['District'].apply(get_hydro_zone)
    panel_df['stage_of_extraction'] = panel_df['District'].map(STAGE_OF_EXTRACTION)
    panel_df['hydro_zone'] = panel_df['District'].map(HYDRO_ZONES).fillna('Unclassified')
    zone_map = {'Unclassified': 0, 'Western': 1, 'Central': 2, 'Bundelkhand': 3, 'Terai': 4}
    panel_df['hydro_zone'] = panel_df['hydro_zone'].map(zone_map)
    
    panel_df['district_year_median'] = panel_df.groupby(['District', 'Year'])['recharge_potential'].transform('median')
    panel_df['target'] = ((panel_df['pre_monsoon_m'] >= 6.0) & (panel_df['pre_monsoon_m'] <= 35.0) & (panel_df['recharge_potential'] > panel_df['district_year_median'])).astype(int)
    return panel_df, locations

panel_df, unique_locations = process_telemetry()

In [ ]:
def get_gee_base_stack():
    dem = ee.ImageCollection('COPERNICUS/DEM/GLO30_2024_1').mosaic().select(0)
    slope = ee.Terrain.slope(dem).rename('slope_deg')
    curvature = dem.convolve(ee.Kernel.laplacian8()).rename('curvature')
    tpi_250 = dem.subtract(dem.focal_mean(radius=250, units='meters')).rename('tpi_250_m')
    tpi_1k = dem.subtract(dem.focal_mean(radius=1000, units='meters')).rename('tpi_1k_m')
    tpi_5k = dem.subtract(dem.focal_mean(radius=5000, units='meters')).rename('tpi_5k_m')
    
    def soil_30_100(asset, prefix, divisor, name):
        img = ee.Image(asset)
        mid = (img.select(f'{prefix}_30-60cm_mean').multiply(30).add(img.select(f'{prefix}_60-100cm_mean').multiply(40)).divide(70))
        return mid.divide(divisor).rename(name)

    clay_0 = ee.Image('projects/soilgrids-isric/clay_mean').select('clay_0-5cm_mean').divide(10).rename('soil_clay_0_5_pct')
    clay_30 = soil_30_100('projects/soilgrids-isric/clay_mean', 'clay', 10, 'soil_clay_30_100_pct')
    sand_0 = ee.Image('projects/soilgrids-isric/sand_mean').select('sand_0-5cm_mean').divide(10).rename('soil_sand_0_5_pct')
    sand_30 = soil_30_100('projects/soilgrids-isric/sand_mean', 'sand', 10, 'soil_sand_30_100_pct')
    soc_0 = ee.Image('projects/soilgrids-isric/soc_mean').select('soc_0-5cm_mean').divide(10).rename('soil_soc_0_5')
    soc_30 = soil_30_100('projects/soilgrids-isric/soc_mean', 'soc', 10, 'soil_soc_30_100')
    
    modis_ndvi = ee.ImageCollection('MODIS/061/MOD13Q1').filterDate('2021-01-01', '2026-01-01').filter(ee.Filter.calendarRange(1, 4, 'month')).mean().select('NDVI').multiply(0.0001).rename('dry_season_ndvi')
    dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate('2021-01-01', '2024-01-01').select('label').mode()
    crop_frac = dw.eq(4).rename('crop_frac')
    built_frac = dw.eq(6).rename('built_frac')
    tree_frac = dw.eq(1).rename('tree_frac')
    
    merit = ee.Image('MERIT/Hydro/v1_0_1')
    upa = merit.select('upa')
    log_upa = upa.add(0.01).log().rename('log_upstream_area')
    hand = merit.select('hnd').rename('hand_m')
    
    # Capped fastDistanceTransform to 128 to dramatically reduce timeout errors!
    dist_to_river = (upa.gt(10).unmask(0).fastDistanceTransform(128).sqrt().multiply(ee.Image.pixelArea().sqrt()).rename('distance_to_river_m'))
    
    rain_clim = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate('1991-01-01', '2020-12-31').filter(ee.Filter.calendarRange(6, 9, 'month')).sum().divide(30).rename('rain_clim_mm')
    
    return ee.Image.cat([slope, curvature, tpi_250, tpi_1k, tpi_5k, clay_0, clay_30, sand_0, sand_30, soc_0, soc_30, modis_ndvi, crop_frac, built_frac, tree_frac, log_upa, hand, dist_to_river, dem.rename('elevation_m'), rain_clim])

print('Initiating GEE Checkpoint Extraction Workflow...')
base_stack = get_gee_base_stack()
yearly_rain = []
for y in range(2021, 2026):
    monsoon = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate(f'{y}-06-01', f'{y}-10-01')
    yearly_rain.extend([monsoon.sum().rename(f'chirps_{y}'), monsoon.map(lambda i: i.gt(2)).sum().rename(f'rainy_days_{y}'), ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate(f'{y}-10-01', f'{y}-11-01').sum().rename(f'late_rain_{y}')])
full_stack = ee.Image.cat([base_stack] + yearly_rain)

done_wells = set(pd.read_csv(GEE_CACHE_FILE)['well_id'].astype(str)) if os.path.exists(GEE_CACHE_FILE) else set()
todo = unique_locations[~unique_locations['well_id'].astype(str).isin(done_wells)]
print(f'{len(done_wells)} wells already done, {len(todo)} to go.')

def extract(df):
    feats = [ee.Feature(ee.Geometry.Point([r.Longitude, r.Latitude]), {'well_id': str(r.well_id)}) for r in df.itertuples()]
    fc = ee.FeatureCollection(feats).map(lambda f: f.buffer(GEE_BUFFER_M))
    for attempt in range(3):
        try:
            res = full_stack.reduceRegions(fc, ee.Reducer.mean(), scale=30, tileScale=8).getInfo()
            return [f['properties'] for f in res['features']]
        except Exception as e:
            msg = str(e).lower()
            if 'did not match' in msg or 'not found' in msg: raise
            if 'timed out' in msg or 'too many' in msg: break
            time.sleep(5 * (attempt + 1))
    if len(df) == 1:
        print('FAILED well', df.iloc[0]['well_id'])
        return []
    mid = len(df) // 2
    return extract(df.iloc[:mid]) + extract(df.iloc[mid:])

if len(todo) > 0:
    chunks = [todo.iloc[i:i+25] for i in range(0, len(todo), 25)]
    with ThreadPoolExecutor(max_workers=4) as ex:
        futs = [ex.submit(extract, c) for c in chunks]
        for n, fut in enumerate(as_completed(futs), 1):
            rows = fut.result()
            if rows:
                pd.DataFrame(rows).to_csv(GEE_CACHE_FILE, mode='a', header=not os.path.exists(GEE_CACHE_FILE), index=False)
            print(f'chunks done {n}/{len(chunks)}', end='\r')

gee = pd.read_csv(GEE_CACHE_FILE).drop_duplicates('well_id')
missing = set(unique_locations['well_id'].astype(str)) - set(gee['well_id'].astype(str))
print(f'\nWells fully missing features: {len(missing)}')

gee['well_id'] = gee['well_id'].astype(str)
panel_df['well_id'] = panel_df['well_id'].astype(str)
static_cols = [c for c in gee.columns if not c.startswith('chirps_') and not c.startswith('rainy_days_') and not c.startswith('late_rain_') and c != 'well_id']
panel_df = panel_df.merge(gee[['well_id'] + static_cols], on='well_id', how='left')

def assign_yr(row, p):
    w = gee[gee['well_id'] == row['well_id']]
    return w.iloc[0][f'{p}_{row["Year"]}'] if len(w)>0 and f'{p}_{row["Year"]}' in w.columns else np.nan

panel_df['monsoon_rainfall_mm'] = panel_df.apply(lambda r: assign_yr(r, 'chirps'), axis=1)
panel_df['rainy_days'] = panel_df.apply(lambda r: assign_yr(r, 'rainy_days'), axis=1)
panel_df['late_rain_mm'] = panel_df.apply(lambda r: assign_yr(r, 'late_rain'), axis=1)
panel_df['rain_anomaly_mm'] = panel_df['monsoon_rainfall_mm'] - panel_df['rain_clim_mm']
panel_df['TWI'] = panel_df['log_upstream_area'] + np.log(900.0) - np.log(np.tan(np.radians(panel_df['slope_deg'])) + 0.001)
# Drop remaining NaNs (failed extracted wells)
panel_df = panel_df.dropna(subset=['monsoon_rainfall_mm', 'rain_clim_mm'])

In [ ]:
from scipy.spatial import cKDTree
from scipy.special import erf
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

print("--- 1. Aggregating Panel & Computing GEC-2015 Specific-Yield Ground Truth ---")

raw_gee_cols = [
    'rain_clim_mm', 'slope_deg', 'curvature', 'tpi_250_m', 'tpi_1k_m', 'tpi_5k_m',
    'soil_clay_0_5_pct', 'soil_clay_30_100_pct', 'soil_sand_0_5_pct', 'soil_sand_30_100_pct',
    'soil_soc_0_5', 'soil_soc_30_100', 'dry_season_ndvi', 'crop_frac', 'built_frac', 'tree_frac',
    'log_upstream_area', 'elevation_m', 'distance_to_river_m', 'hand_m', 'TWI',
    'hydro_zone', 'pdf_hydro_zone', 'stage_of_extraction'
]

# Extract multi-year stability & active 2021-2025 monsoon telemetry from panel_df
panel_tmp = panel_df.copy()
sy_panel_soil = 0.04 + 0.0020 * pd.to_numeric(panel_tmp['soil_sand_30_100_pct'], errors='coerce').fillna(35.0) - 0.0012 * pd.to_numeric(panel_tmp['soil_clay_30_100_pct'], errors='coerce').fillna(25.0)
sy_panel_fac = np.where(panel_tmp['pdf_hydro_zone'] == 2, 0.45, np.where(panel_tmp['pdf_hydro_zone'] == 0, 1.15, 1.0))
panel_tmp['sy_est'] = np.clip(sy_panel_soil * sy_panel_fac, 0.02, 0.22)
panel_tmp['yr_eff_rech'] = panel_tmp['recharge_potential'] * panel_tmp['sy_est']
panel_tmp['yr_headroom_ok'] = ((panel_tmp['pre_monsoon_m'] >= 5.0) & (panel_tmp['pre_monsoon_m'] <= 35.0)).astype(float)

well_df = panel_tmp.groupby('well_id').agg(
    Latitude=('Latitude', 'first'),
    Longitude=('Longitude', 'first'),
    District=('District', 'first'),
    n_obs_years=('Year', 'count'),
    pre_monsoon_m=('pre_monsoon_m', 'median'),
    pre_monsoon_iqr=('pre_monsoon_m', lambda x: float(np.percentile(x, 75) - np.percentile(x, 25))),
    post_monsoon_m=('post_monsoon_m', 'median'),
    recharge_potential=('recharge_potential', 'median'),
    monsoon_rainfall_mm=('monsoon_rainfall_mm', 'median'),
    rainy_days=('rainy_days', 'median'),
    late_rain_mm=('late_rain_mm', 'median'),
    rain_anomaly_mm=('rain_anomaly_mm', 'median'),
    emp_headroom_rate=('yr_headroom_ok', 'mean'),
    **{c: (c, 'first') for c in raw_gee_cols if c in panel_tmp.columns}
).reset_index()

rain_dyn_cols = ['monsoon_rainfall_mm', 'rainy_days', 'late_rain_mm', 'rain_anomaly_mm']
for col in raw_gee_cols + rain_dyn_cols:
    if col in well_df.columns:
        well_df[col] = pd.to_numeric(well_df[col], errors='coerce')
        well_df[col] = well_df[col].fillna(well_df[col].median())

def norm_cdf(x):
    """Vectorized standard normal cumulative distribution function."""
    return 0.5 * (1.0 + erf(np.asarray(x, dtype=float) / np.sqrt(2.0)))

def weighted_quantile(values, weights, q=0.5):
    """Outlier-resistant weighted quantile for hydro-geomorphic kriging."""
    sorter = np.argsort(values)
    v = values[sorter]
    w = weights[sorter]
    cdf = np.cumsum(w) / np.sum(w)
    return float(np.interp(q, cdf, v))

def add_engineered_features(df):
    df = df.copy()
    for c in raw_gee_cols + rain_dyn_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(well_df[c].median())

    # Specific Yield (Sy) estimation from SoilGrids texture & hydrogeological regime (GEC-2015 norms)
    sy_soil = 0.04 + 0.0020 * df['soil_sand_30_100_pct'] - 0.0012 * df['soil_clay_30_100_pct']
    sy_factor = np.where(df['pdf_hydro_zone'] == 2, 0.45, np.where(df['pdf_hydro_zone'] == 0, 1.15, 1.0))
    df['specific_yield_est'] = np.clip(sy_soil * sy_factor, 0.02, 0.22)

    # Physical Vadose, Topographic & 2021-2025 Monsoon Forcing Interactions
    df['permeability_ratio_surface'] = df['soil_sand_0_5_pct'] / (df['soil_clay_0_5_pct'] + 1.0)
    df['permeability_ratio_subsurface'] = df['soil_sand_30_100_pct'] / (df['soil_clay_30_100_pct'] + 1.0)
    df['clay_vertical_gradient'] = df['soil_clay_30_100_pct'] - df['soil_clay_0_5_pct']
    df['sand_vertical_gradient'] = df['soil_sand_30_100_pct'] - df['soil_sand_0_5_pct']
    df['tpi_convergence_diff'] = df['tpi_250_m'] - df['tpi_5k_m']
    df['tpi_local_vs_mid'] = df['tpi_250_m'] - df['tpi_1k_m']
    df['infiltration_capacity_idx'] = (df['rain_clim_mm'] * df['permeability_ratio_surface']) / (df['slope_deg'] + 0.5)
    df['hand_to_river_ratio'] = df['hand_m'] / (df['distance_to_river_m'] + 10.0)
    df['runoff_capture_idx'] = df['TWI'] * np.log1p(df['rain_clim_mm']) / (df['slope_deg'] + 1.0)
    df['stress_to_rain_ratio'] = df['stage_of_extraction'] / (df['rain_clim_mm'] + 1.0)
    df['vadose_storage_proxy'] = df['hand_m'] * df['specific_yield_est']
    df['alluvial_recharge_idx'] = (df['rain_clim_mm'] * df['soil_sand_30_100_pct']) / (df['soil_clay_30_100_pct'] + 5.0)
    df['drainage_density_proxy'] = df['log_upstream_area'] / np.log1p(df['distance_to_river_m'] + 10.0)

    # Dynamic 2021-2025 Monsoon & October Late-Rain Percolation Features
    df['rain_intensity_mm_day'] = df['monsoon_rainfall_mm'] / (df['rainy_days'] + 1.0)
    df['late_rain_fraction'] = df['late_rain_mm'] / (df['monsoon_rainfall_mm'] + 1.0)
    df['effective_monsoon_soak_idx'] = (df['monsoon_rainfall_mm'] * np.sqrt(df['rainy_days'] + 1.0) * df['permeability_ratio_surface']) / (df['slope_deg'] + 1.0)
    df['late_rain_retention_idx'] = df['late_rain_mm'] * df['permeability_ratio_subsurface'] / (df['specific_yield_est'] + 0.02)
    return df

well_df = add_engineered_features(well_df)

# Store fixed GEC-2015 physical thresholds for training & live inference
VOL_RECHARGE_THRESH = float((well_df['recharge_potential'] * well_df['specific_yield_est']).quantile(0.45))
PERM_SUBSURFACE_THRESH = float(well_df['permeability_ratio_subsurface'].quantile(0.35))
well_df['site_required_rise_m'] = VOL_RECHARGE_THRESH / np.clip(well_df['specific_yield_est'], 0.02, 0.22)

# Multi-year empirical volumetric recharge pass rate per well
panel_tmp['yr_vol_ok'] = (panel_tmp['yr_eff_rech'] >= VOL_RECHARGE_THRESH).astype(float)
vol_rates = panel_tmp.groupby('well_id')['yr_vol_ok'].mean().rename('emp_vol_rate')
well_df = well_df.merge(vol_rates, on='well_id', how='left')
well_df['emp_vol_rate'] = well_df['emp_vol_rate'].fillna(0.5)

def compute_surface_feasibility_gate(df):
    """Evaluates observable GEE surface/lithology screening criteria (1 = Feasible, 0 = Infeasible)."""
    perm_ok = (df['permeability_ratio_subsurface'] >= PERM_SUBSURFACE_THRESH) & (df['built_frac'] < 0.5)
    slope_ok = np.where(df['pdf_hydro_zone'] == 2, df['slope_deg'] <= 7.0, df['slope_deg'] <= 5.0)
    return (perm_ok & slope_ok).astype(float)

well_df['effective_recharge_m'] = well_df['recharge_potential'] * well_df['specific_yield_est']
well_df['rif_observed'] = np.clip(well_df['effective_recharge_m'] / (well_df['monsoon_rainfall_mm'] / 1000.0 + 0.1), -0.2, 0.8)
well_df['headroom_ok'] = ((well_df['pre_monsoon_m'] >= 5.0) & (well_df['pre_monsoon_m'] <= 35.0)).astype(int)
well_df['vol_recharge_ok'] = (well_df['effective_recharge_m'] >= VOL_RECHARGE_THRESH).astype(int)
well_df['subsurface_ok'] = ((well_df['headroom_ok'] == 1) & (well_df['vol_recharge_ok'] == 1)).astype(int)
well_df['surface_gate'] = compute_surface_feasibility_gate(well_df)
well_df['target'] = ((well_df['subsurface_ok'] == 1) & (well_df['surface_gate'] == 1.0)).astype(int)

print(f"Unique Wells: {len(well_df)} | Subsurface-Suitable: {well_df['subsurface_ok'].sum()} ({well_df['subsurface_ok'].mean():.1%}) | Final GEC-2015 Suitable: {well_df['target'].sum()} ({well_df['target'].mean():.1%})")

SATELLITE_DRIFT_COLS = [
    'elevation_m', 'rain_clim_mm', 'monsoon_rainfall_mm', 'rainy_days', 'late_rain_mm', 'rain_anomaly_mm',
    'rain_intensity_mm_day', 'late_rain_fraction', 'effective_monsoon_soak_idx', 'late_rain_retention_idx',
    'slope_deg', 'curvature', 'tpi_250_m', 'tpi_1k_m', 'tpi_5k_m',
    'soil_clay_0_5_pct', 'soil_clay_30_100_pct', 'soil_sand_0_5_pct', 'soil_sand_30_100_pct',
    'soil_soc_0_5', 'soil_soc_30_100', 'dry_season_ndvi', 'crop_frac', 'built_frac', 'tree_frac',
    'log_upstream_area', 'distance_to_river_m', 'hand_m', 'TWI',
    'hydro_zone', 'pdf_hydro_zone', 'stage_of_extraction',
    'specific_yield_est', 'permeability_ratio_surface', 'permeability_ratio_subsurface',
    'clay_vertical_gradient', 'sand_vertical_gradient', 'tpi_convergence_diff', 'tpi_local_vs_mid',
    'infiltration_capacity_idx', 'hand_to_river_ratio', 'runoff_capture_idx',
    'stress_to_rain_ratio', 'vadose_storage_proxy', 'alluvial_recharge_idx', 'drainage_density_proxy'
]

def fit_satellite_drift_models(train_df):
    """Fits Block-Purged out-of-fold satellite physical trend models for Drift-Transferred Co-Kriging."""
    coords = train_df[['Latitude', 'Longitude']].values
    blocks = train_df['spatial_block'].values
    tree = cKDTree(np.radians(coords))
    r = 3.5 / 6371.0

    X_sat = train_df[SATELLITE_DRIFT_COLS]
    y_pre = train_df['pre_monsoon_m'].values
    y_rech = train_df['recharge_potential'].values
    y_sub = train_df['subsurface_ok'].values

    oof_pre = np.full(len(train_df), np.nan)
    oof_rech = np.full(len(train_df), np.nan)

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    for tr, te in sgkf.split(X_sat, y_sub, blocks):
        conflicts = set()
        for idx in te:
            conflicts.update(tree.query_ball_point(np.radians(coords[idx]), r))
        tr_clean = np.array([i for i in tr if i not in conflicts])

        m_pre_cb = CatBoostRegressor(iterations=360, depth=4, learning_rate=0.03, l2_leaf_reg=15, verbose=0, allow_writing_files=False, random_seed=42).fit(X_sat.iloc[tr_clean], y_pre[tr_clean])
        m_pre_lgb = LGBMRegressor(n_estimators=260, max_depth=4, num_leaves=15, learning_rate=0.03, reg_lambda=12.0, random_state=42, verbose=-1).fit(X_sat.iloc[tr_clean], y_pre[tr_clean])
        m_pre_rd = make_pipeline(SimpleImputer(strategy='median'), RobustScaler(), Ridge(alpha=50.0)).fit(X_sat.iloc[tr_clean], y_pre[tr_clean])
        oof_pre[te] = 0.50 * m_pre_cb.predict(X_sat.iloc[te]) + 0.30 * m_pre_lgb.predict(X_sat.iloc[te]) + 0.20 * m_pre_rd.predict(X_sat.iloc[te])

        m_rech_cb = CatBoostRegressor(iterations=360, depth=4, learning_rate=0.03, l2_leaf_reg=15, verbose=0, allow_writing_files=False, random_seed=42).fit(X_sat.iloc[tr_clean], y_rech[tr_clean])
        m_rech_lgb = LGBMRegressor(n_estimators=260, max_depth=4, num_leaves=15, learning_rate=0.03, reg_lambda=12.0, random_state=42, verbose=-1).fit(X_sat.iloc[tr_clean], y_rech[tr_clean])
        m_rech_rd = make_pipeline(SimpleImputer(strategy='median'), RobustScaler(), Ridge(alpha=50.0)).fit(X_sat.iloc[tr_clean], y_rech[tr_clean])
        oof_rech[te] = 0.50 * m_rech_cb.predict(X_sat.iloc[te]) + 0.30 * m_rech_lgb.predict(X_sat.iloc[te]) + 0.20 * m_rech_rd.predict(X_sat.iloc[te])

    full_pre_cb = CatBoostRegressor(iterations=400, depth=4, learning_rate=0.03, l2_leaf_reg=15, verbose=0, allow_writing_files=False, random_seed=42).fit(X_sat, y_pre)
    full_pre_lgb = LGBMRegressor(n_estimators=300, max_depth=4, num_leaves=15, learning_rate=0.03, reg_lambda=12.0, random_state=42, verbose=-1).fit(X_sat, y_pre)
    full_pre_rd = make_pipeline(SimpleImputer(strategy='median'), RobustScaler(), Ridge(alpha=50.0)).fit(X_sat, y_pre)

    full_rech_cb = CatBoostRegressor(iterations=400, depth=4, learning_rate=0.03, l2_leaf_reg=15, verbose=0, allow_writing_files=False, random_seed=42).fit(X_sat, y_rech)
    full_rech_lgb = LGBMRegressor(n_estimators=300, max_depth=4, num_leaves=15, learning_rate=0.03, reg_lambda=12.0, random_state=42, verbose=-1).fit(X_sat, y_rech)
    full_rech_rd = make_pipeline(SimpleImputer(strategy='median'), RobustScaler(), Ridge(alpha=50.0)).fit(X_sat, y_rech)

    drift_models = {
        'pre_cb': full_pre_cb, 'pre_lgb': full_pre_lgb, 'pre_rd': full_pre_rd,
        'rech_cb': full_rech_cb, 'rech_lgb': full_rech_lgb, 'rech_rd': full_rech_rd
    }
    return oof_pre, oof_rech, drift_models

def predict_satellite_drift(df, drift_models):
    X_sat = df[SATELLITE_DRIFT_COLS]
    sat_pre = 0.50 * drift_models['pre_cb'].predict(X_sat) + 0.30 * drift_models['pre_lgb'].predict(X_sat) + 0.20 * drift_models['pre_rd'].predict(X_sat)
    sat_rech = 0.50 * drift_models['rech_cb'].predict(X_sat) + 0.30 * drift_models['rech_lgb'].predict(X_sat) + 0.20 * drift_models['rech_rd'].predict(X_sat)
    return sat_pre, sat_rech

# --- 2. Multi-Scale Physical Co-Kriging (Near-Field >=4.5km + Cross-Block Meso-Scale >=9.0km) ---
def compute_purged_subsurface_idw(target_df, ref_df, min_dist_km=4.5, k=24):
    target_coords = target_df[['Latitude', 'Longitude']].values
    ref_coords = ref_df[['Latitude', 'Longitude']].values

    tree = cKDTree(np.radians(ref_coords))
    dists_rad, idxs = tree.query(np.radians(target_coords), k=min(k + 65, len(ref_coords)))
    dists_km = dists_rad * 6371.0

    ref_pre = ref_df['pre_monsoon_m'].values
    ref_rech = ref_df['recharge_potential'].values
    ref_sat_pre = ref_df['sat_pred_pre_m'].values
    ref_sat_rech = ref_df['sat_pred_rech_m'].values
    ref_res_pre = np.clip(ref_pre - ref_sat_pre, -12.0, 12.0)
    ref_res_rech = np.clip(ref_rech - ref_sat_rech, -2.5, 2.5)
    ref_rif = ref_df['rif_observed'].values
    ref_eff = ref_df['effective_recharge_m'].values
    ref_emp_head = ref_df['emp_headroom_rate'].values
    ref_emp_vol = ref_df['emp_vol_rate'].values
    ref_elev = ref_df['elevation_m'].values
    ref_hand = ref_df['hand_m'].values
    ref_twi = ref_df['TWI'].values
    ref_perm = ref_df['permeability_ratio_subsurface'].values
    ref_rain = ref_df['monsoon_rainfall_mm'].values
    ref_zone = ref_df['pdf_hydro_zone'].values
    ref_rel = np.sqrt(np.clip(ref_df['n_obs_years'].values, 1.0, 6.0))

    t_elev = target_df['elevation_m'].values
    t_hand = target_df['hand_m'].values
    t_twi = target_df['TWI'].values
    t_perm = target_df['permeability_ratio_subsurface'].values
    t_rain = target_df['monsoon_rainfall_mm'].values
    t_zone = target_df['pdf_hydro_zone'].values
    t_sy = target_df['specific_yield_est'].values
    t_gate = target_df['surface_gate'].values
    t_sat_pre = target_df['sat_pred_pre_m'].values
    t_sat_rech = target_df['sat_pred_rech_m'].values

    rows = []
    for i in range(len(target_coords)):
        # Scale 1: Near-Field Kernel (>= min_dist_km)
        valid = dists_km[i] >= min_dist_km
        if not np.any(valid):
            valid = dists_km[i] > 0.001
        d = dists_km[i][valid][:k]
        ix = idxs[i][valid][:k]

        dz = (t_elev[i] - ref_elev[ix]) / 14.0
        dh = (t_hand[i] - ref_hand[ix]) / 4.5
        dt = (t_twi[i] - ref_twi[ix]) / 2.5
        dp = (t_perm[i] - ref_perm[ix]) / 1.8
        zone_match = np.where(t_zone[i] == ref_zone[ix], 1.0, 0.35)

        sim_penalty = 1.0 + 0.40 * dz ** 2 + 0.45 * dh ** 2 + 0.20 * dt ** 2 + 0.20 * dp ** 2
        w = (zone_match * ref_rel[ix]) / ((d ** 1.55 + 3.0) * sim_penalty)
        w /= w.sum()

        # Scale 2: Meso-Scale Cross-Block Kernel (>= 9.5 km, invariant between training & holdout blocks)
        valid_meso = dists_km[i] >= max(min_dist_km, 9.5)
        if np.sum(valid_meso) < 6:
            valid_meso = valid
        d_m = dists_km[i][valid_meso][:k]
        ix_m = idxs[i][valid_meso][:k]
        dz_m = (t_elev[i] - ref_elev[ix_m]) / 18.0
        dh_m = (t_hand[i] - ref_hand[ix_m]) / 6.0
        w_reg = (np.where(t_zone[i] == ref_zone[ix_m], 1.0, 0.40) * ref_rel[ix_m]) / ((d_m + 8.0) * (1.0 + 0.30 * dz_m ** 2 + 0.30 * dh_m ** 2))
        w_reg /= w_reg.sum()

        # 1. Pre-Monsoon Water Table Depth (Drift-Transferred, RK, & Weighted Median)
        implied_pre = ref_pre[ix] + 0.55 * (t_sat_pre[i] - ref_sat_pre[ix]) + 0.20 * np.clip(t_elev[i] - ref_elev[ix], -10.0, 10.0)
        implied_pre_m = ref_pre[ix_m] + 0.55 * (t_sat_pre[i] - ref_sat_pre[ix_m]) + 0.20 * np.clip(t_elev[i] - ref_elev[ix_m], -10.0, 10.0)
        rk_pre = float(np.clip(t_sat_pre[i] + np.sum(w * ref_res_pre[ix]), 0.5, 60.0))
        rk_pre_reg = float(np.clip(t_sat_pre[i] + np.sum(w_reg * ref_res_pre[ix_m]), 0.5, 60.0))
        est_pre = float(np.sum(w * ref_pre[ix]))
        med_pre = weighted_quantile(implied_pre, w, 0.50)
        med_pre_reg = weighted_quantile(implied_pre_m, w_reg, 0.50)
        std_pre = float(np.sqrt(np.sum(w * (implied_pre - med_pre) ** 2) + 0.85 ** 2))

        # 2. Monsoon Water-Table Rise (Rainfall-Scaled, Drift-Transferred, & Weighted Median)
        rain_scale = np.clip((t_rain[i] + 50.0) / (ref_rain[ix] + 50.0), 0.65, 1.45)
        implied_rech = ref_rech[ix] * (0.50 + 0.50 * rain_scale) + 0.50 * (t_sat_rech[i] - ref_sat_rech[ix])
        implied_rech_m = ref_rech[ix_m] + 0.50 * (t_sat_rech[i] - ref_sat_rech[ix_m])
        rk_rech = float(np.clip(t_sat_rech[i] + np.sum(w * ref_res_rech[ix]), -0.5, 15.0))
        rk_rech_reg = float(np.clip(t_sat_rech[i] + np.sum(w_reg * ref_res_rech[ix_m]), -0.5, 15.0))
        est_rech = float(np.sum(w * ref_rech[ix]))
        med_rech = weighted_quantile(implied_rech, w, 0.50)
        med_rech_reg = weighted_quantile(implied_rech_m, w_reg, 0.50)
        q75_rech = weighted_quantile(implied_rech, w, 0.75)
        std_rech = float(np.sqrt(np.sum(w * (implied_rech - med_rech) ** 2) + 0.25 ** 2))

        kriged_rif = float(0.65 * np.sum(w * ref_rif[ix]) + 0.35 * np.sum(w_reg * ref_rif[ix_m]))
        rif_eff_recharge = float(kriged_rif * (t_rain[i] / 1000.0))

        # 3. Headroom Probabilities (Blending Near-Field & Meso-Scale Cross-Block Estimates)
        blend_pre = 0.35 * rk_pre + 0.30 * med_pre + 0.20 * rk_pre_reg + 0.15 * med_pre_reg
        neigh_head_prob = norm_cdf((implied_pre - 5.0) / 0.80) - norm_cdf((implied_pre - 35.0) / 1.5)
        neigh_head_meso = norm_cdf((implied_pre_m - 5.0) / 0.90) - norm_cdf((implied_pre_m - 35.0) / 1.5)
        idw_headroom_prob = float(0.50 * np.sum(w * neigh_head_prob) + 0.25 * np.sum(w_reg * neigh_head_meso) + 0.25 * np.sum(w * ref_emp_head[ix]))
        kriged_headroom_prob = float(norm_cdf((blend_pre - 5.0) / std_pre) - norm_cdf((blend_pre - 35.0) / std_pre))
        waterlogged_risk = float(np.sum(w * norm_cdf((5.0 - implied_pre) / 0.80)))

        # 4. Site-Coupled Volumetric Recharge Probabilities (using THIS site's exact 30m Sy_i)
        req_rise_i = VOL_RECHARGE_THRESH / max(t_sy[i], 0.02)
        blend_rech = 0.35 * rk_rech + 0.30 * med_rech + 0.20 * rk_rech_reg + 0.15 * med_rech_reg
        site_coupled_eff = 0.70 * (blend_rech * t_sy[i]) + 0.30 * rif_eff_recharge
        site_coupled_eff_reg = rk_rech_reg * t_sy[i]
        site_rise_surplus = (site_coupled_eff / max(t_sy[i], 0.02)) - req_rise_i
        site_rise_zscore = site_rise_surplus / std_rech

        neigh_vol_prob_coupled = norm_cdf((implied_rech - req_rise_i) / 0.32)
        neigh_vol_prob_meso = norm_cdf((implied_rech_m - req_rise_i) / 0.35)
        idw_site_vol_prob = float(0.50 * np.sum(w * neigh_vol_prob_coupled) + 0.25 * np.sum(w_reg * neigh_vol_prob_meso) + 0.25 * np.sum(w * ref_emp_vol[ix]))
        kriged_site_vol_prob = float(norm_cdf(site_rise_zscore / 0.85))

        # 5. Analytic Joint Physics Prior (Product + Fuzzy-AND Min)
        neigh_joint_coupled = float(np.sum(w * (neigh_head_prob * neigh_vol_prob_coupled)))
        p_h_prior = 0.55 * idw_headroom_prob + 0.45 * kriged_headroom_prob
        p_v_prior = 0.55 * idw_site_vol_prob + 0.45 * kriged_site_vol_prob
        fuzzy_and_prior = float(min(p_h_prior, p_v_prior))
        analytic_physics_prior = float(t_gate[i] * (0.65 * (p_h_prior * p_v_prior) + 0.35 * fuzzy_and_prior))

        rows.append({
            'rk_pre_monsoon_m': rk_pre,
            'rk_pre_monsoon_regional': rk_pre_reg,
            'idw_pre_monsoon_m': est_pre,
            'med_pre_monsoon_m': med_pre,
            'med_pre_monsoon_regional': med_pre_reg,
            'idw_pre_monsoon_std': std_pre,
            'waterlogged_risk': waterlogged_risk,
            'idw_headroom_prob': idw_headroom_prob,
            'kriged_headroom_prob': kriged_headroom_prob,
            'rk_recharge_m': rk_rech,
            'rk_recharge_regional': rk_rech_reg,
            'idw_recharge_m': est_rech,
            'med_recharge_m': med_rech,
            'med_recharge_regional': med_rech_reg,
            'q75_recharge_m': q75_rech,
            'idw_recharge_std': std_rech,
            'rif_eff_recharge_m': rif_eff_recharge,
            'site_coupled_eff_recharge_m': site_coupled_eff,
            'site_coupled_eff_regional_m': site_coupled_eff_reg,
            'site_rise_surplus_m': site_rise_surplus,
            'site_rise_zscore': site_rise_zscore,
            'idw_site_vol_prob': idw_site_vol_prob,
            'kriged_site_vol_prob': kriged_site_vol_prob,
            'idw_coupled_joint_prob': neigh_joint_coupled,
            'fuzzy_and_prior': fuzzy_and_prior,
            'analytic_physics_prior': analytic_physics_prior,
            'idw_effective_recharge_m': float(np.sum(w * ref_eff[ix])),
            'local_elevation_anomaly_m': float(t_elev[i] - np.sum(w * ref_elev[ix]))
        })

    return pd.DataFrame(rows, index=target_df.index)

# --- 3. Stratified Spatial Block Holdout Vault (Exact Same 162 Holdout Wells) ---
well_df['spatial_block'] = (
    (np.floor(well_df['Latitude'] / 0.35).astype(int)).astype(str) + "_" +
    (np.floor(well_df['Longitude'] / 0.35).astype(int)).astype(str)
)
block_summary = well_df.groupby(['spatial_block', 'pdf_hydro_zone']).agg(
    n_wells=('well_id', 'count'), pos_rate=('target', 'mean')
).reset_index()

holdout_blocks = []
np.random.seed(42)
for zone in [0, 1, 2]:
    b_zone = block_summary[(block_summary['pdf_hydro_zone'] == zone) & (block_summary['n_wells'] >= 5) & (block_summary['pos_rate'] > 0)]
    if len(b_zone) > 0:
        n_pick = max(1, int(0.18 * len(b_zone)))
        holdout_blocks.extend(np.random.choice(b_zone['spatial_block'], size=n_pick, replace=False).tolist())

raw_holdout_mask = well_df['spatial_block'].isin(holdout_blocks)
panel_holdout = well_df[raw_holdout_mask].reset_index(drop=True)

# Purge any training well within 3.0 km of ANY holdout well to guarantee 0% spatial leakage
holdout_tree = cKDTree(np.radians(panel_holdout[['Latitude', 'Longitude']].values))
train_candidates = well_df[~raw_holdout_mask].reset_index(drop=True)
leaked = holdout_tree.query_ball_point(np.radians(train_candidates[['Latitude', 'Longitude']].values), r=3.0 / 6371.0)
keep_train = [len(x) == 0 for x in leaked]
panel_train_val = train_candidates[keep_train].reset_index(drop=True)

oof_sat_pre, oof_sat_rech, cv_drift_models = fit_satellite_drift_models(panel_train_val)
panel_train_val['sat_pred_pre_m'] = oof_sat_pre
panel_train_val['sat_pred_rech_m'] = oof_sat_rech

hold_sat_pre, hold_sat_rech = predict_satellite_drift(panel_holdout, cv_drift_models)
panel_holdout['sat_pred_pre_m'] = hold_sat_pre
panel_holdout['sat_pred_rech_m'] = hold_sat_rech

train_idw = compute_purged_subsurface_idw(panel_train_val, panel_train_val, min_dist_km=5.0)
holdout_idw = compute_purged_subsurface_idw(panel_holdout, panel_train_val, min_dist_km=3.0)

for col in train_idw.columns:
    panel_train_val[col] = train_idw[col].values
    panel_holdout[col] = holdout_idw[col].values

print(f"Purged Training Wells: {len(panel_train_val)} (Positives: {panel_train_val['target'].sum()}) | Purged Holdout Wells: {len(panel_holdout)} (Positives: {panel_holdout['target'].sum()})")

DEPLOYABLE_FEATURES = [
    'surface_gate', 'analytic_physics_prior', 'fuzzy_and_prior', 'idw_coupled_joint_prob',
    'idw_headroom_prob', 'kriged_headroom_prob', 'waterlogged_risk',
    'idw_site_vol_prob', 'kriged_site_vol_prob',
    'site_required_rise_m', 'rif_eff_recharge_m', 'site_coupled_eff_recharge_m', 'site_coupled_eff_regional_m',
    'site_rise_surplus_m', 'site_rise_zscore',
    'sat_pred_pre_m', 'sat_pred_rech_m',
    'rk_pre_monsoon_m', 'rk_pre_monsoon_regional', 'idw_pre_monsoon_m', 'med_pre_monsoon_m', 'med_pre_monsoon_regional', 'idw_pre_monsoon_std',
    'rk_recharge_m', 'rk_recharge_regional', 'idw_recharge_m', 'med_recharge_m', 'med_recharge_regional', 'q75_recharge_m', 'idw_recharge_std',
    'idw_effective_recharge_m', 'local_elevation_anomaly_m',
    'monsoon_rainfall_mm', 'rainy_days', 'late_rain_mm', 'rain_anomaly_mm',
    'rain_intensity_mm_day', 'late_rain_fraction', 'effective_monsoon_soak_idx', 'late_rain_retention_idx',
    'specific_yield_est', 'permeability_ratio_surface', 'permeability_ratio_subsurface',
    'clay_vertical_gradient', 'sand_vertical_gradient', 'tpi_convergence_diff', 'tpi_local_vs_mid',
    'infiltration_capacity_idx', 'runoff_capture_idx', 'hand_to_river_ratio',
    'stress_to_rain_ratio', 'vadose_storage_proxy', 'alluvial_recharge_idx', 'drainage_density_proxy',
    'stage_of_extraction', 'elevation_m', 'rain_clim_mm', 'slope_deg', 'curvature', 'tpi_250_m', 'tpi_1k_m', 'tpi_5k_m',
    'soil_clay_0_5_pct', 'soil_clay_30_100_pct', 'soil_sand_0_5_pct', 'soil_sand_30_100_pct',
    'soil_soc_0_5', 'soil_soc_30_100', 'dry_season_ndvi', 'crop_frac', 'built_frac', 'tree_frac',
    'log_upstream_area', 'distance_to_river_m', 'hand_m', 'TWI',
    'hydro_zone', 'pdf_hydro_zone'
]

X_cv = panel_train_val[DEPLOYABLE_FEATURES].copy()
y_cv = panel_train_val['target'].values
y_sub_cv = panel_train_val['subsurface_ok'].values
scale_weight = float(np.sum(y_sub_cv == 0) / max(1, np.sum(y_sub_cv == 1)))



In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, KFold
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, roc_auc_score, f1_score, fbeta_score, classification_report
from scipy.optimize import minimize
import optuna

print("\n--- 1. Block-Purged Spatial CV Splitter ---")
class SpatialPurgeCV:
    """Stratified Spatial-Block K-Fold with 3.5 km KDTree border purging to match Holdout Block geometry."""
    def __init__(self, n_splits=5, buffer_km=3.5, coords=None, blocks=None):
        self.n_splits = n_splits
        self.buffer_km = buffer_km
        self.coords = coords
        self.blocks = blocks

    def split(self, X, y=None, groups=None):
        latlon = self.coords if self.coords is not None else X[['Latitude', 'Longitude']].values
        tree = cKDTree(np.radians(latlon))
        r = self.buffer_km / 6371.0
        indices = np.arange(len(latlon))
        grp = self.blocks if self.blocks is not None else groups

        if grp is not None and y is not None and len(np.unique(y)) > 1 and len(np.unique(grp)) >= self.n_splits:
            splitter = StratifiedGroupKFold(n_splits=self.n_splits, shuffle=True, random_state=42).split(indices, y, grp)
        elif y is not None and len(np.unique(y)) > 1:
            splitter = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=42).split(indices, y)
        else:
            splitter = KFold(n_splits=self.n_splits, shuffle=True, random_state=42).split(indices)

        for tr, te in splitter:
            conflicts = set()
            for idx in te:
                conflicts.update(tree.query_ball_point(np.radians(latlon[idx]), r))
            tr_clean = np.array([i for i in tr if i not in conflicts])
            yield tr_clean, te

spatial_coords_cv = panel_train_val[['Latitude', 'Longitude']].values
spatial_blocks_cv = panel_train_val['spatial_block'].values
spatial_splits_cv = list(
    SpatialPurgeCV(n_splits=5, buffer_km=3.5, coords=spatial_coords_cv, blocks=spatial_blocks_cv).split(X_cv, y_cv)
)

print("\n--- 2. Optuna Hyperparameter Search (Block-Purged Pure Subsurface CV) ---")
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 320, 600),
        'depth': trial.suggest_int('depth', 4, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.015, 0.04, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 8.0, 35.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.72, 0.90),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.68, 0.90),
        'verbose': 0,
        'allow_writing_files': False,
        'bootstrap_type': 'Bernoulli',
        'random_seed': 42,
        'scale_pos_weight': scale_weight
    }
    
    oof_probs = np.zeros(len(X_cv))
    valid_mask = np.zeros(len(X_cv), dtype=bool)
    X_sub = X_cv.drop(columns=['surface_gate'], errors='ignore')
    
    for tr, te in spatial_splits_cv:
        if len(np.unique(y_sub_cv[tr])) < 2 or len(np.unique(y_cv[te])) < 2:
            continue
        base = CatBoostClassifier(**params)
        base.fit(X_sub.iloc[tr], y_sub_cv[tr])
        gate = X_cv.iloc[te]['surface_gate'].values
        prior = X_cv.iloc[te]['analytic_physics_prior'].values
        p_cb = base.predict_proba(X_sub.iloc[te])[:, 1]
        oof_probs[te] = gate * np.sqrt(np.clip(p_cb * prior, 0.0, 1.0))
        valid_mask[te] = True
        
    return roc_auc_score(y_cv[valid_mask], oof_probs[valid_mask])

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=16)
print("Best Optuna Parameters:", study.best_params)

cb_params = study.best_params.copy()
cb_params['verbose'] = 0
cb_params['allow_writing_files'] = False
cb_params['bootstrap_type'] = 'Bernoulli'
cb_params['scale_pos_weight'] = scale_weight

class FactorizedPhysicsStackingEnsemble(BaseEstimator, ClassifierMixin):
    """
    Multiplicative 3-Stack Physics-Guided Ensemble:
      1. Full 5-Model Stacking Ensemble + Continuous Depth Regressor for P(Headroom OK)
      2. Full 5-Model Stacking Ensemble + Continuous Rise Regressor for P(Volumetric Recharge OK)
      3. Full 5-Model Stacking Ensemble + Conditional Specialist for P(Subsurface OK)
      4. Multiplicative Geometric & Fuzzy-AND Fusion guarantees that a site cannot achieve high
         probability unless P(Headroom OK), P(Volumetric Recharge OK), and P(Subsurface OK)
         all simultaneously agree.
    """
    def __init__(self, splits, pos_weight, cb_config, vol_thresh, fusion_weights=None):
        self.splits = splits
        self.pos_weight = pos_weight
        self.cb_config = cb_config
        self.vol_thresh = vol_thresh
        self.fusion_weights = fusion_weights

    def _augment_stage1_features(self, X, pred_pre, pred_rech):
        X_aug = X.copy()
        sy = X_aug['specific_yield_est'].values
        req_rise = self.vol_thresh / np.clip(sy, 0.02, 0.22)
        pred_eff = pred_rech * sy

        p_head_reg = norm_cdf((pred_pre - 5.0) / 1.1) - norm_cdf((pred_pre - 35.0) / 2.0)
        p_vol_reg = norm_cdf((pred_rech - req_rise) / 0.30)

        X_aug['stage1_pred_pre_m'] = pred_pre
        X_aug['stage1_pred_rech_m'] = pred_rech
        X_aug['stage1_pred_eff_m'] = pred_eff
        X_aug['stage1_prob_headroom'] = p_head_reg
        X_aug['stage1_prob_vol'] = p_vol_reg
        X_aug['stage1_prob_joint'] = X_aug['surface_gate'].values * p_head_reg * p_vol_reg
        return X_aug

    def _make_5model_stack(self, pos_w=1.0, seed=42):
        cb_clf = CatBoostClassifier(**{**self.cb_config, 'scale_pos_weight': pos_w}, random_seed=seed)
        xgb_clf = XGBClassifier(
            n_estimators=360, max_depth=4, learning_rate=0.025,
            subsample=0.82, colsample_bytree=0.78, reg_lambda=12.0,
            scale_pos_weight=pos_w, random_state=seed, eval_metric="logloss"
        )
        lgb_clf = LGBMClassifier(
            n_estimators=360, max_depth=4, num_leaves=15, learning_rate=0.025,
            subsample=0.82, colsample_bytree=0.78, reg_lambda=12.0,
            scale_pos_weight=pos_w, random_state=seed, verbose=-1
        )
        rf_clf = make_pipeline(
            SimpleImputer(strategy='median'),
            RandomForestClassifier(n_estimators=320, max_depth=8, min_samples_leaf=4, class_weight='balanced_subsample', random_state=seed, n_jobs=-1)
        )
        et_clf = make_pipeline(
            SimpleImputer(strategy='median'),
            ExtraTreesClassifier(n_estimators=320, max_depth=8, min_samples_leaf=4, class_weight='balanced_subsample', random_state=seed, n_jobs=-1)
        )
        meta = make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler(),
            LogisticRegression(penalty='l2', C=0.20, solver='lbfgs', max_iter=1000)
        )
        return StackingClassifier(
            estimators=[('cb', cb_clf), ('xgb', xgb_clf), ('lgb', lgb_clf), ('rf', rf_clf), ('et', et_clf)],
            final_estimator=meta,
            cv=self.splits,
            stack_method='predict_proba',
            n_jobs=-1
        )

    def fit(self, X, y, pre_m=None, rech_m=None, y_headroom=None, y_vol_recharge=None, y_subsurface=None):
        self.classes_ = np.array([0, 1])
        X_clean = X.reset_index(drop=True)
        y_sub_target = y_subsurface if y_subsurface is not None else y

        # Stage 1: Out-of-fold Continuous Subsurface Regressors (Depth & Rise)
        if pre_m is not None and rech_m is not None:
            self.has_stage1_ = True
            oof_pre = np.full(len(X_clean), np.nan)
            oof_rech = np.full(len(X_clean), np.nan)

            for tr, te in self.splits:
                r_pre_cb = CatBoostRegressor(iterations=350, depth=4, learning_rate=0.03, l2_leaf_reg=12, verbose=0, allow_writing_files=False, random_seed=42).fit(X_clean.iloc[tr], pre_m[tr])
                r_pre_lgb = LGBMRegressor(n_estimators=260, max_depth=4, num_leaves=15, learning_rate=0.03, reg_lambda=10.0, random_state=42, verbose=-1).fit(X_clean.iloc[tr], pre_m[tr])
                oof_pre[te] = 0.60 * r_pre_cb.predict(X_clean.iloc[te]) + 0.40 * r_pre_lgb.predict(X_clean.iloc[te])

                r_rech_cb = CatBoostRegressor(iterations=350, depth=4, learning_rate=0.03, l2_leaf_reg=12, verbose=0, allow_writing_files=False, random_seed=42).fit(X_clean.iloc[tr], rech_m[tr])
                r_rech_lgb = LGBMRegressor(n_estimators=260, max_depth=4, num_leaves=15, learning_rate=0.03, reg_lambda=10.0, random_state=42, verbose=-1).fit(X_clean.iloc[tr], rech_m[tr])
                oof_rech[te] = 0.60 * r_rech_cb.predict(X_clean.iloc[te]) + 0.40 * r_rech_lgb.predict(X_clean.iloc[te])

            oof_pre = np.where(np.isnan(oof_pre), np.nanmedian(pre_m), oof_pre)
            oof_rech = np.where(np.isnan(oof_rech), np.nanmedian(rech_m), oof_rech)

            self.reg_pre_cb_ = CatBoostRegressor(iterations=380, depth=4, learning_rate=0.03, l2_leaf_reg=12, verbose=0, allow_writing_files=False, random_seed=42).fit(X_clean, pre_m)
            self.reg_pre_lgb_ = LGBMRegressor(n_estimators=280, max_depth=4, num_leaves=15, learning_rate=0.03, reg_lambda=10.0, random_state=42, verbose=-1).fit(X_clean, pre_m)
            self.reg_rech_cb_ = CatBoostRegressor(iterations=380, depth=4, learning_rate=0.03, l2_leaf_reg=12, verbose=0, allow_writing_files=False, random_seed=42).fit(X_clean, rech_m)
            self.reg_rech_lgb_ = LGBMRegressor(n_estimators=280, max_depth=4, num_leaves=15, learning_rate=0.03, reg_lambda=10.0, random_state=42, verbose=-1).fit(X_clean, rech_m)

            X_stage2 = self._augment_stage1_features(X_clean, oof_pre, oof_rech)
        else:
            self.has_stage1_ = False
            X_stage2 = X_clean

        X_no_gate = X_stage2.drop(columns=['surface_gate'], errors='ignore')

        # Stack 1 & Stack 2: Full 5-Model Stacking Classifiers for P(Headroom OK) and P(Volumetric Recharge OK)
        if y_headroom is not None and y_vol_recharge is not None:
            self.has_factorized_ = True
            self.head_stack_ = self._make_5model_stack(pos_w=1.0, seed=42).fit(X_no_gate, y_headroom)
            self.vol_stack_ = self._make_5model_stack(pos_w=1.0, seed=100).fit(X_no_gate, y_vol_recharge)
        else:
            self.has_factorized_ = False

        # Stack 3A: Conditional Surface-Feasible Specialist (trained strictly on surface_gate == 1 sites, uninflated pos_weight=1.15)
        f_mask = (X_stage2['surface_gate'].values == 1.0)
        if np.sum(f_mask) >= 50 and len(np.unique(y[f_mask])) > 1:
            self.has_cond_specialist_ = True
            X_f = X_no_gate.iloc[f_mask]
            y_f = y[f_mask]
            self.cond_cb_ = CatBoostClassifier(iterations=420, depth=4, learning_rate=0.025, l2_leaf_reg=14, scale_pos_weight=1.15, verbose=0, allow_writing_files=False, random_seed=42).fit(X_f, y_f)
            self.cond_lgb_ = LGBMClassifier(n_estimators=320, max_depth=4, num_leaves=15, learning_rate=0.025, subsample=0.82, colsample_bytree=0.78, reg_lambda=12.0, scale_pos_weight=1.15, random_state=42, verbose=-1).fit(X_f, y_f)
            self.cond_xgb_ = XGBClassifier(n_estimators=320, max_depth=4, learning_rate=0.025, subsample=0.82, colsample_bytree=0.78, reg_lambda=12.0, scale_pos_weight=1.15, random_state=42, eval_metric="logloss").fit(X_f, y_f)
        else:
            self.has_cond_specialist_ = False

        # Stack 3B: All-Well 5-Model Stacking Classifier trained on uncorrupted subsurface_ok (uninflated pos_weight=1.15)
        self.sub_stack_ = self._make_5model_stack(pos_w=1.15, seed=2024).fit(X_no_gate, y_sub_target)
        return self

    def predict_components(self, X):
        """Returns [p_fact, p_fuzzy_min, p_joint_sub, p_prior, p_head, p_vol] and surface_gate."""
        X_clean = X.reset_index(drop=True)
        if getattr(self, 'has_stage1_', False):
            pred_pre = 0.60 * self.reg_pre_cb_.predict(X_clean) + 0.40 * self.reg_pre_lgb_.predict(X_clean)
            pred_rech = 0.60 * self.reg_rech_cb_.predict(X_clean) + 0.40 * self.reg_rech_lgb_.predict(X_clean)
            X_stage2 = self._augment_stage1_features(X_clean, pred_pre, pred_rech)
        else:
            X_stage2 = X_clean

        X_no_gate = X_stage2.drop(columns=['surface_gate'], errors='ignore')
        gate = X_clean['surface_gate'].values if 'surface_gate' in X_clean.columns else np.ones(len(X_clean))
        p_prior = X_clean['analytic_physics_prior'].values if 'analytic_physics_prior' in X_clean.columns else np.full(len(X_clean), 0.2)
        p_sub_stack = self.sub_stack_.predict_proba(X_no_gate)[:, 1]

        if getattr(self, 'has_cond_specialist_', False):
            p_cond = (
                0.45 * self.cond_cb_.predict_proba(X_no_gate)[:, 1] +
                0.30 * self.cond_lgb_.predict_proba(X_no_gate)[:, 1] +
                0.25 * self.cond_xgb_.predict_proba(X_no_gate)[:, 1]
            )
            p_joint_sub = 0.55 * p_sub_stack + 0.45 * p_cond
        else:
            p_joint_sub = p_sub_stack

        if getattr(self, 'has_factorized_', False):
            p_head_clf = self.head_stack_.predict_proba(X_no_gate)[:, 1]
            p_vol_clf = self.vol_stack_.predict_proba(X_no_gate)[:, 1]

            p_head = (
                0.55 * p_head_clf +
                0.25 * X_stage2['stage1_prob_headroom'].values +
                0.20 * (0.55 * X_clean['idw_headroom_prob'].values + 0.45 * X_clean['kriged_headroom_prob'].values)
            )
            p_vol = (
                0.55 * p_vol_clf +
                0.25 * X_stage2['stage1_prob_vol'].values +
                0.20 * (0.55 * X_clean['idw_site_vol_prob'].values + 0.45 * X_clean['kriged_site_vol_prob'].values)
            )
            p_fact = p_head * p_vol
            p_fuzzy_min = np.minimum(p_head, p_vol)
        else:
            p_fact = p_joint_sub
            p_fuzzy_min = p_joint_sub

        # Note: p_joint_sub is multiplicatively coupled with sqrt(p_fact) so it can never bypass the headroom/recharge veto!
        p_coupled_sub = np.sqrt(np.clip(p_joint_sub * p_fact, 0.0, 1.0))
        p_coupled_min = np.sqrt(np.clip(p_joint_sub * p_fuzzy_min, 0.0, 1.0))

        return np.column_stack([p_fact, p_coupled_sub, p_coupled_min, p_prior]), gate

    def predict_proba(self, X):
        comps, gate = self.predict_components(X)
        w = self.fusion_weights if self.fusion_weights is not None else np.array([0.35, 0.35, 0.15, 0.15])
        w = np.asarray(w, dtype=float) / np.sum(w)

        # Blend in both geometric (log-probability) and arithmetic spaces—every component is already multiplicatively gated!
        p_geom = np.exp(np.dot(np.log(np.clip(comps, 1e-4, 1.0)), w))
        p_arith = np.dot(comps, w)
        p_subsurface = 0.55 * p_geom + 0.45 * p_arith

        p_final = np.clip(p_subsurface * gate, 0.0, 1.0)
        return np.column_stack([1.0 - p_final, p_final])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

print("\n--- 3. Block-Purged Out-Of-Fold Evaluation & Multiplicative Weight Calibration ---")
pre_m_cv = panel_train_val['pre_monsoon_m'].values
rech_m_cv = panel_train_val['recharge_potential'].values
y_head_cv = panel_train_val['headroom_ok'].values
y_vol_cv = panel_train_val['vol_recharge_ok'].values

oof_comps = np.zeros((len(X_cv), 4))
oof_gates = np.zeros(len(X_cv))
valid_oof_mask = np.zeros(len(X_cv), dtype=bool)

for tr, te in spatial_splits_cv:
    inner_coords = panel_train_val.iloc[tr][['Latitude', 'Longitude']].values
    inner_blocks = panel_train_val.iloc[tr]['spatial_block'].values
    inner_splits = list(
        SpatialPurgeCV(n_splits=4, buffer_km=3.5, coords=inner_coords, blocks=inner_blocks).split(X_cv.iloc[tr], y_sub_cv[tr])
    )
    fold_model = FactorizedPhysicsStackingEnsemble(
        splits=inner_splits, pos_weight=scale_weight, cb_config=cb_params, vol_thresh=VOL_RECHARGE_THRESH
    )
    fold_model.fit(
        X_cv.iloc[tr], y_cv[tr],
        pre_m=pre_m_cv[tr], rech_m=rech_m_cv[tr],
        y_headroom=y_head_cv[tr], y_vol_recharge=y_vol_cv[tr],
        y_subsurface=y_sub_cv[tr]
    )
    c_te, gate_te = fold_model.predict_components(X_cv.iloc[te])
    oof_comps[te] = c_te
    oof_gates[te] = gate_te
    valid_oof_mask[te] = True

y_eval = y_cv[valid_oof_mask]
c_eval = oof_comps[valid_oof_mask]
gate_eval = oof_gates[valid_oof_mask]

f_eval_mask = (gate_eval == 1.0)
def loss_func(weights):
    w = np.asarray(weights) / np.sum(weights)
    p_g = np.exp(np.dot(np.log(np.clip(c_eval[f_eval_mask], 1e-4, 1.0)), w))
    p_a = np.dot(c_eval[f_eval_mask], w)
    p = np.clip(0.55 * p_g + 0.45 * p_a, 0.0, 1.0)
    return brier_score_loss(y_eval[f_eval_mask], p) - 0.25 * roc_auc_score(y_eval[f_eval_mask], p)

res_opt = minimize(
    loss_func,
    x0=np.array([0.35, 0.35, 0.15, 0.15]),
    bounds=[(0.12, 0.45)] * 4,
    method='L-BFGS-B'
)
opt_weights = res_opt.x / np.sum(res_opt.x)
print("Optimal Multiplicative Weights [Fact(H*V), CoupledSub, CoupledMin, Prior]:", np.round(opt_weights, 3))

p_g_oof = np.exp(np.dot(np.log(np.clip(c_eval, 1e-4, 1.0)), opt_weights))
p_a_oof = np.dot(c_eval, opt_weights)
oof_eval = np.clip((0.55 * p_g_oof + 0.45 * p_a_oof) * gate_eval, 0.0, 1.0)

thresholds = np.linspace(0.20, 0.80, 61)
f1_scores = [f1_score(y_eval, (oof_eval >= t).astype(int)) for t in thresholds]
opt_t_f1 = float(thresholds[np.argmax(f1_scores)])

high_prec_candidates = [
    (t, fbeta_score(y_eval, (oof_eval >= t).astype(int), beta=0.5))
    for t in thresholds if t >= max(0.42, opt_t_f1 + 0.07)
]
opt_t = float(max(high_prec_candidates, key=lambda x: x[1])[0])

oof_auc = roc_auc_score(y_eval, oof_eval)
print(f"Training-pool Block-Purged Spatial OOF AUC: {oof_auc:.4f}")
print(f"Balanced Threshold (Max F1): {opt_t_f1:.4f} | High-Precision Threshold (Max F0.5): {opt_t:.4f}")

print("\n--- 4. Final Holdout Vault Evaluation (3-Stack Multiplicative Physics Ensemble) ---")
eval_stack = FactorizedPhysicsStackingEnsemble(
    splits=spatial_splits_cv, pos_weight=scale_weight, cb_config=cb_params,
    vol_thresh=VOL_RECHARGE_THRESH, fusion_weights=opt_weights
)
eval_stack.fit(
    X_cv, y_cv,
    pre_m=pre_m_cv, rech_m=rech_m_cv,
    y_headroom=y_head_cv, y_vol_recharge=y_vol_cv,
    y_subsurface=y_sub_cv
)

X_holdout = panel_holdout[DEPLOYABLE_FEATURES].copy()
y_holdout = panel_holdout['target'].values
holdout_probs = eval_stack.predict_proba(X_holdout)[:, 1]

holdout_auc = roc_auc_score(y_holdout, holdout_probs)
brier = brier_score_loss(y_holdout, holdout_probs)
print(f">> TRUE ULTIMATE HOLDOUT ROC-AUC (Stacking): {holdout_auc:.4f}")
print(f">> Brier Score: {brier:.4f}")

for pct in (0.10, 0.20):
    k_cnt = max(1, int(pct * len(holdout_probs)))
    k_idx = np.argsort(holdout_probs)[::-1][:k_cnt]
    print(f">> Precision @ Top-{int(pct*100)}%: {np.mean(y_holdout[k_idx]):.4f}")

print("\n[Balanced F1 Threshold Classification Report]:")
print(classification_report(y_holdout, (holdout_probs >= opt_t_f1).astype(int), target_names=['Unsuitable', 'Suitable']))
print("[High-Precision F0.5 Threshold Classification Report]:")
print(classification_report(y_holdout, (holdout_probs >= opt_t).astype(int), target_names=['Unsuitable', 'Suitable']))

In [ ]:
import shap
import joblib
import matplotlib.pyplot as plt

print("\n--- Building Calibrated Production Stacking Ensemble (FULL WELL DATASET) ---")
oof_sat_pre_full, oof_sat_rech_full, full_drift_models = fit_satellite_drift_models(well_df)
well_df['sat_pred_pre_m'] = oof_sat_pre_full
well_df['sat_pred_rech_m'] = oof_sat_rech_full

full_idw = compute_purged_subsurface_idw(well_df, well_df, min_dist_km=5.0)
for col in full_idw.columns:
    well_df[col] = full_idw[col].values

X_full = well_df[DEPLOYABLE_FEATURES].copy()
y_full = well_df['target'].values
y_sub_full = well_df['subsurface_ok'].values
pre_m_full = well_df['pre_monsoon_m'].values
rech_m_full = well_df['recharge_potential'].values
y_head_full = well_df['headroom_ok'].values
y_vol_full = well_df['vol_recharge_ok'].values
full_coords = well_df[['Latitude', 'Longitude']].values
full_blocks = well_df['spatial_block'].values
full_spatial_splits = list(
    SpatialPurgeCV(n_splits=5, buffer_km=3.5, coords=full_coords, blocks=full_blocks).split(X_full, y_sub_full)
)

scale_weight_full = float(np.sum(y_sub_full == 0) / max(1, np.sum(y_sub_full == 1)))

production_stack = FactorizedPhysicsStackingEnsemble(
    splits=full_spatial_splits, pos_weight=scale_weight_full, cb_config=cb_params,
    vol_thresh=VOL_RECHARGE_THRESH, fusion_weights=opt_weights
)
print("Fitting 3-Stack Multiplicative Physics Ensemble on 100% of unique wells...")
production_stack.fit(
    X_full, y_full,
    pre_m=pre_m_full, rech_m=rech_m_full,
    y_headroom=y_head_full, y_vol_recharge=y_vol_full,
    y_subsurface=y_sub_full
)

reference_cols = [
    'Latitude', 'Longitude', 'spatial_block', 'n_obs_years', 'pre_monsoon_m', 'recharge_potential',
    'monsoon_rainfall_mm', 'rainy_days', 'late_rain_mm', 'rain_anomaly_mm',
    'sat_pred_pre_m', 'sat_pred_rech_m', 'rif_observed', 'effective_recharge_m',
    'emp_headroom_rate', 'emp_vol_rate', 'subsurface_ok', 'target',
    'elevation_m', 'hand_m', 'TWI', 'permeability_ratio_subsurface', 'pdf_hydro_zone'
]
artifact = {
    'model': production_stack,
    'drift_models': full_drift_models,
    'features': DEPLOYABLE_FEATURES,
    'satellite_drift_cols': SATELLITE_DRIFT_COLS,
    'stage_of_extraction_map': STAGE_OF_EXTRACTION,
    'optimal_threshold': opt_t,
    'balanced_threshold': opt_t_f1,
    'vol_recharge_thresh': VOL_RECHARGE_THRESH,
    'perm_subsurface_thresh': PERM_SUBSURFACE_THRESH,
    'reference_df': well_df[reference_cols].copy(),
    'version': VERSION_ID
}
joblib.dump(artifact, f'/kaggle/working/{VERSION_ID}.pkl')
print(f"✅ Production Artifact exported to: /kaggle/working/{VERSION_ID}.pkl")

print("\n--- Civic Explainability: SHAP Analysis on Pure Subsurface CatBoost Estimator ---")
X_full_no_gate = X_full.drop(columns=['surface_gate'], errors='ignore')
cb_explainer_model = CatBoostClassifier(**{**cb_params, 'scale_pos_weight': scale_weight_full}, random_seed=42)
cb_explainer_model.fit(X_full_no_gate, y_sub_full)
explainer = shap.TreeExplainer(cb_explainer_model)
shap_values = explainer.shap_values(X_full_no_gate)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_full_no_gate, show=False)
plt.title("SHAP Summary: Subsurface Hydrogeological Feature Attribution (UP)")
plt.savefig('/kaggle/working/shap_summary.png', bbox_inches='tight')
plt.show()
print("Saved SHAP plot to /kaggle/working/shap_summary.png")



In [ ]:
# ==========================================
# INFERENCE API FOR DEPLOYMENT
# ==========================================
def predict_site_recharge(lat, lon, district_name):
    print(f"Running Live Inference for ({lat}, {lon}) in {district_name}...")
    point = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lon, lat]).buffer(GEE_BUFFER_M))])
    stack = get_gee_base_stack()
    res = stack.reduceRegions(collection=point, reducer=ee.Reducer.mean(), scale=30).getInfo()
    props = res['features'][0]['properties']
    
    df = pd.DataFrame([props])
    df['Latitude'] = lat
    df['Longitude'] = lon
    df['TWI'] = df['log_upstream_area'] + np.log(900.0) - np.log(np.tan(np.radians(df['slope_deg'])) + 0.001)
    
    d_norm = normalize_district(district_name)
    bhabar_terai = ['saharanpur', 'bijnor', 'moradabad', 'rampur', 'bareilly', 'pilibhit', 'lakhimpur kheri', 'bahraich', 'shravasti', 'balrampur', 'siddharthnagar', 'mahrajganj']
    bundelkhand = ['jhansi', 'lalitpur', 'jalaun', 'hamirpur', 'mahoba', 'banda', 'chitrakoot', 'mirzapur', 'sonbhadra']
    df['pdf_hydro_zone'] = 0 if d_norm in bhabar_terai else 2 if d_norm in bundelkhand else 1
    df['stage_of_extraction'] = STAGE_OF_EXTRACTION.get(d_norm, 70.0)
    
    zone_str = HYDRO_ZONES.get(d_norm, 'Unclassified')
    zone_map = {'Unclassified': 0, 'Western': 1, 'Central': 2, 'Bundelkhand': 3, 'Terai': 4}
    df['hydro_zone'] = zone_map.get(zone_str, 0)
    
    # Interpolate 2021-2025 monsoon & late-rain telemetry from nearest reference stations
    ref_df = artifact['reference_df']
    ref_tree = cKDTree(np.radians(ref_df[['Latitude', 'Longitude']].values))
    d_rad, ix = ref_tree.query(np.radians([[lat, lon]]), k=8)
    w_r = 1.0 / ((d_rad[0] * 6371.0) ** 1.5 + 1.0)
    w_r /= w_r.sum()
    for r_col in ['monsoon_rainfall_mm', 'rainy_days', 'late_rain_mm', 'rain_anomaly_mm']:
        df[r_col] = float(np.sum(w_r * ref_df[r_col].values[ix]))
    
    df = add_engineered_features(df)
    df['site_required_rise_m'] = VOL_RECHARGE_THRESH / np.clip(df['specific_yield_est'], 0.02, 0.22)
    df['surface_gate'] = compute_surface_feasibility_gate(df)
    
    sat_pre, sat_rech = predict_satellite_drift(df, artifact['drift_models'])
    df['sat_pred_pre_m'] = sat_pre
    df['sat_pred_rech_m'] = sat_rech
    
    live_idw = compute_purged_subsurface_idw(df, ref_df, min_dist_km=0.0)
    for col in live_idw.columns:
        df[col] = live_idw[col].values
        
    prob = float(production_stack.predict_proba(df[DEPLOYABLE_FEATURES])[0, 1])
    
    return {
        'Probability_Suitable': round(prob, 4),
        'Is_Suitable_High_Precision': bool(prob >= opt_t),
        'Is_Suitable_Balanced': bool(prob >= opt_t_f1),
        'Surface_Feasible': bool(df['surface_gate'].iloc[0] == 1.0),
        'High_Precision_Threshold': round(opt_t, 4),
        'Balanced_Threshold': round(opt_t_f1, 4),
        'Extracted_Elevation': round(float(df['elevation_m'].iloc[0]), 2)
    }

